In [ ]:
import numpy as np
from scipy.ndimage import uniform_filter, sobel

class LogisticRegressionScratch:
    def __init__(self, lr=0.5, n_iters=2000, l2=1e-3):
        self.lr, self.n_iters, self.l2 = lr, n_iters, l2
        self.w, self.b, self.loss_history = None, 0.0, []

    @staticmethod
    def _sigmoid(z):
        out = np.empty_like(z)
        pos = z >= 0
        out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
        exp_z = np.exp(z[~pos])
        out[~pos] = exp_z / (1.0 + exp_z)
        return out

    def _bce_loss(self, y, p):
        p = np.clip(p, 1e-12, 1.0 - 1e-12)
        loss = -np.mean(y*np.log(p) + (1-y)*np.log(1-p))
        return loss + (self.l2 / (2*len(y))) * np.sum(self.w**2)

    def fit(self, X, y):
        m, d = X.shape; self.w, self.b = np.zeros(d), 0.0
        for it in range(self.n_iters):
            p = self._sigmoid(X @ self.w + self.b)
            error = p - y
            grad_w = (X.T @ error)/m + (self.l2/m)*self.w
            grad_b = np.mean(error)
            self.w -= self.lr * grad_w; self.b -= self.lr * grad_b
            self.loss_history.append(self._bce_loss(y, p))
        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# Feature Extraction (Sobel + 3x3 Texture Statistics)
def extract_pixel_features(gray):
    gx = sobel(gray, axis=1); gy = sobel(gray, axis=0)
    grad_mag = np.sqrt(gx**2 + gy**2)
    local_mean = uniform_filter(gray, size=3)
    local_sq = uniform_filter(gray**2, size=3)
    local_std = np.sqrt(np.clip(local_sq - local_mean**2, 0, None))
    return np.stack([gray, grad_mag, gx, gy, local_mean, local_std], axis=-1)

# Feature Standardization & Performance Metrics
class StandardScaler:
    def fit(self, X):
        self.mean, self.std = X.mean(axis=0), X.std(axis=0) + 1e-8
        return self
    def transform(self, X):
        return (X - self.mean) / self.std

def classification_report(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2*prec*rec / (prec + rec) if (prec + rec) else 0.0
    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1,
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
